In [6]:
"""
b3_accent.py — B3: a NEW phoneme built in the token tables, used for every occurrence in a text.

Put this file next to functions.py, accent_tools.py and b4_accent.py (it reuses b4_accent's player page).
Edit the SETTINGS and run   python b3_accent.py
or in a notebook:           from b3_accent import run;  run(variants=[dict(label="t+ʲ", to={"ʲ": 1}, combine="add")])

HOW IT WORKS
  Kokoro has two lookup tables (178 rows): prosody 178×128 and acoustic 178×512. Only 114 rows are used.
  For every variant and weight, a new row is written into a FREE slot of BOTH tables and registered
  under a new one-character symbol (Ⓐ, Ⓑ, …). Every SUB_FROM in the text is replaced by that symbol,
  and Kokoro reads it like any other phoneme: the encoders interpret it in context, durations / pitch /
  energy are predicted for it — there is no extra token and no [j] segment.

  combine="interpolate":  row = (1-w)·FROM + w·TO          (an in-between phoneme; length kept)
  combine="add":          row = FROM + w·TO                (FROM fully kept, TO's information ADDED on top;
                                                            the rows are nearly orthogonal codes, so the sum
                                                            carries both: good for t + ʲ = one palatalized t)
  TO is a recipe: {"ʨ": 1} or {"ʲ": 1} or {"j": 0.5, "ʲ": 0.5} …
  prosody_row="same":  the prosody table gets the same mix: durations, pitch, energy AND VOICING follow
                       the new sound (needed for t→n, t→d: voicing comes from the pitch/energy curves)
  prosody_row="from":  the prosody table keeps FROM's row: timing and voicing of the original t
                       (only the spectrum changes; t→n then still sounds t-like)

  GEMINATES: "tː" is two tokens, t + ː. Replacing only the t gives Ⓒː, and the ː frames (which hold the
  RELEASE of a geminate stop) are read as a lengthened t -> manipulated closure, clean t burst. GEMINATE
  "merge" replaces the whole tː by one new token with the frames of both (needs TIMING="original" to keep
  the length); "double" writes two new tokens.

  SELF-CHECK: for interpolate, w = 1, prosody "same" and a single-symbol TO, the new phoneme IS that symbol,
  so the audio must equal writing TO in the text directly. The script renders that and prints the result.
  If it says False, the new symbol is not reaching the model (e.g. _encode uses its own symbol map).

  The rows are codes the model never saw (it was trained on 114 symbols), so the result can be anything
  from "t" to something odd — that is why several variants and weights go on one page.
  All table changes are undone after run() (and before each new run).

OUTPUT (OUT_DIR/b3_<from>_variants.html): original + one player per variant × weight, and a wav for each.
"""
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

from accent_tools import load_story, manipulate_with_quota, make_alignment, block_changes, voiced_mask
import functions as FN
from functions import model, VOCAB, SR, align_phonemes, _encode, _voice_pack, transfer_durations
from b4_accent import _player_fragment, _page, _name

# =====================================================================================================
# SETTINGS
# =====================================================================================================
TEXT = """Tutti i turisti tornano tardi in albergo; Tommaso taglia la torta al cioccolato.
Il tetto della torre è tutto storto; Teresa telefona tre volte alla settimana.
Tante tazze di tè sono sul tavolo; Il tram attraversa tutta la città in tempo.
Ottobre porta temporali e tempeste; Matteo mette i tegami sotto il tavolo.
Stanotte tutti guardano le stelle; Il gatto tranquillo sta sul tappeto tutto il pomeriggio."""     # the text itself, or a path to a .txt file

# TEXT = """La casa di Carlo è dietro il parco; Il cane corre sul campo.
# Chiara compra un chilo di pane; Questo libro costa poco.
# Marco beve un caffè caldo; La scuola è chiusa per il cattivo tempo.
# Il cuoco prepara la carne; Luca ha comprato una borsa nera per Carla.
# Che bella giornata per andare in campagna con Paolo; Quanto costa il biglietto per Como?"""


VOICE = "if_sara"
SUB_FROM = "t"                # every occurrence of this phoneme gets the new symbol
TO = 'ɖ'  #subs = ['ð','ɖ', 'ʨ', 'ʦ']
# each variant: label, to (recipe), combine, weights, and optionally prosody_row / keep_norm
VARIANTS = [
    # dict(label="t + w·ʲ (palatal code added)", to={"ʲ": 1.0}, combine="add", weights=[0.3, 0.6, 1.0]),
    # dict(label="t + w·j (glide code added)", to={"j": 1.0}, combine="add", weights=[0.3, 0.6]),
    dict(label=f"t → {TO} (in-between)", to={TO: 1.0}, combine="interpolate", weights=[ 0.6, 0.7, 0.8, 0.9, 1.0]),
]
PROSODY_ROW = "same"          # default for variants: "same" (mix the prosody row too: timing, voicing and
                              # energy follow the new sound) | "from" (keep FROM's prosody row: timing,
                              # pitch and VOICING of the original t; a voiced target like n or d then
                              # cannot become voiced, because voicing comes from the pitch/energy curves)
GEMINATE = "merge"            # geminates (tː): "merge" -> ONE new token with the frames of t AND ː
                              #   (the release at the end is the new phoneme too) | "double" -> two new tokens
                              #   (ⒸⒸ) sharing those frames | "keep" -> Ⓒː (the ː frames stay t-like)
TIMING = "original"           # "original": every token keeps the frames it has in the ORIGINAL text (a merged
                              #   geminate gets t + ː) | "model": Kokoro predicts the timing of the new text
MAKE_REFERENCE = True
NORMALIZE = None
SPEED = 1.0
SEED = 0
OUT_DIR = "b3_output"
SYMBOLS = [chr(0x24B6 + k) for k in range(26)]   # Ⓐ … Ⓩ: placeholder symbols for the new phonemes


# =====================================================================================================
# TABLE EDITING (undone afterwards)
# =====================================================================================================
def _tables():
    return model.bert.embeddings.word_embeddings.weight, model.text_encoder.embedding.weight


def _clear_caches():
    """functions.py caches encoder outputs, durations and audio PER PHONEME STRING (_ENC_CACHE,
    _DUR_CACHE, _AUDIO_CACHE). The new symbols (Ⓐ, Ⓑ, …) are reused across runs with DIFFERENT table
    rows, so those caches must be emptied whenever the tables change; otherwise _encode returns the
    encoding of an earlier run."""
    if hasattr(FN, "clear_synth_cache"):
        FN.clear_synth_cache()
    for name in ("_ENC_CACHE", "_DUR_CACHE", "_AUDIO_CACHE"):
        c = getattr(FN, name, None)
        if hasattr(c, "clear"):
            c.clear()
    for fn in (getattr(FN, "_encode", None), getattr(FN, "_durations_cached", None), _encode):
        if hasattr(fn, "cache_clear"):
            fn.cache_clear()


class NewPhonemes:
    """Context manager: register new symbols in free rows of both tables; restore everything on exit."""
    def __init__(self):
        self.vocabs = [model.vocab] + ([VOCAB] if VOCAB is not model.vocab else [])
        self.saved, self.added = {}, []

    def __enter__(self):
        _clear_caches()
        return self

    def free_rows(self):
        used = set(model.vocab.values()) | {0}
        return [i for i in range(_tables()[1].shape[0]) if i not in used]

    @torch.no_grad()
    def add(self, symbol, row_pros, row_ac):
        assert len(symbol) == 1 and symbol not in model.vocab, symbol
        i = self.free_rows()[0]
        for tab, row in zip(_tables(), (row_pros, row_ac)):
            self.saved.setdefault((id(tab), i), (tab, tab[i].clone()))
            tab[i] = row
        for v in self.vocabs:
            v[symbol] = i
        self.added.append(symbol)
        _clear_caches()
        return i

    @torch.no_grad()
    def __exit__(self, *exc):
        for (tid, i), (tab, row) in self.saved.items():
            tab[i] = row
        for s in self.added:
            for v in self.vocabs:
                v.pop(s, None)
        _clear_caches()
        return False


def recipe(table, rec):
    return sum(w * table[model.vocab[s]].detach() for s, w in rec.items())


def make_row(table, frm, to, w, combine, keep_norm):
    a, b = table[model.vocab[frm]].detach(), recipe(table, to)
    if combine == "interpolate":
        if w in (0, 1):                          # exact endpoints (no float noise from the rescaling)
            return (a if w == 0 else b).clone()
        v = (1 - w) * a + w * b
        if keep_norm and w > 0:
            v = v * ((1 - w) * a.norm() + w * b.norm()) / v.norm().clamp_min(1e-8)
    elif combine == "add":
        v = a + w * b
        if keep_norm:
            v = v * a.norm() / v.norm().clamp_min(1e-8)
    else:
        raise ValueError(f"combine must be 'interpolate' or 'add', not {combine!r}")
    return v


# =====================================================================================================
# SYNTHESIS
# =====================================================================================================
@torch.no_grad()
def predict_dur(ps, pack, speed=1.0, ref_len=None):
    e = _encode(ps)
    s = pack[(ref_len or len(ps)) - 1].to(model.device)[:, 128:]
    d = model.predictor.text_encoder(e["d_en"], s, e["L"], e["tm"])
    x, _ = model.predictor.lstm(d)
    dur = torch.round(torch.sigmoid(model.predictor.duration_proj(x)).sum(-1) / speed).clamp(min=1).long()
    return dur.reshape(-1).tolist()


@torch.no_grad()
def render_sentence(ps, pack, speed=1.0, seed=0, dur=None, ref_len=None):
    """Kokoro steps on the string (new symbols included). dur = forced durations (None = predicted);
    ref_len = string length that picks the voice row (default len(ps); pass the ORIGINAL length so a
    merged geminate does not change the voice row). Returns audio, durations, F0, N."""
    e = _encode(ps)
    ref = pack[(ref_len or len(ps)) - 1].to(model.device)
    s = ref[:, 128:]
    d = model.predictor.text_encoder(e["d_en"], s, e["L"], e["tm"])
    if dur is None:
        x, _ = model.predictor.lstm(d)
        dur = torch.round(torch.sigmoid(model.predictor.duration_proj(x)).sum(-1) / speed).clamp(min=1).long()
        dur = dur.reshape(-1).tolist()
    aln = make_alignment(ps, dur)
    F0, N = model.predictor.F0Ntrain(d.transpose(-1, -2) @ aln, s)
    torch.manual_seed(seed)
    a = model.decoder(e["t_en"] @ aln, F0, N, ref[:, :128]).squeeze().cpu().numpy()
    f = lambda z: z.squeeze().cpu().numpy()
    return a, dur, f(F0), f(N)


def _render_text(texts, masks, refs, pack, voice_name, label, speed, seed, orig=None, origins=None,
                 timing="model"):
    """texts: IPA per sentence (possibly with new symbols); refs: F0/N of the original for the dashed curves.
    timing="original": durations of the ORIGINAL sentence (orig) mapped onto texts via origins
    (functions.transfer_durations: a group of new tokens gets exactly the frames of what it replaced)."""
    segments, seg_words, seg_altered = [], [], []
    T, F0o, F0r, Ne, Nr, sents, audios = [], [], [], [], [], [], []
    t0, word_off = 0.0, 0
    for ci, (ps, mask) in enumerate(zip(texts, masks)):
        if not ps.strip():
            continue
        dur_in, ref_len = None, None
        if orig is not None:
            ref_len = len(orig[ci])
            if timing == "original":
                dur_in = transfer_durations(orig[ci], ps, origins[ci] if origins is not None else None,
                                            predict_dur(orig[ci], pack, speed), 0.3)
        a, dur, f0, n = render_sentence(ps, pack, speed, seed + ci, dur=dur_in, ref_len=ref_len)
        f0r, nr = refs[ci] if refs is not None else (f0, n)
        segs, widx, altd = align_phonemes(ps, dur, n_samples=len(a), sr=SR, altered_mask=mask)
        segments += [(lab, s0 + t0, s1 + t0, info) for lab, s0, s1, info in segs]
        seg_words += [None if x is None else x + word_off for x in widx]
        seg_altered += altd
        real = [x for x in widx if x is not None]
        word_off += (max(real) + 1) if real else 0
        k = len(f0)
        T += [t0 + np.arange(k) * (len(a) / k) / SR, np.array([t0 + len(a) / SR])]
        kr = len(f0r)                                             # reference may have another length
        F0o += [np.where(voiced_mask(f0), f0, np.nan), np.array([np.nan])]
        F0r += [np.interp(np.arange(k), np.arange(kr) * (k / kr), np.where(voiced_mask(f0r), f0r, np.nan)),
                np.array([np.nan])]
        Ne += [n, np.array([np.nan])]
        Nr += [np.interp(np.arange(k), np.arange(kr) * (k / kr), nr), np.array([np.nan])]
        sents.append(dict(chunk=ci, start=t0, end=t0 + len(a) / SR, donor=None, donor_idx=0))
        audios.append(a); t0 += len(a) / SR
    pros = dict(t=np.concatenate(T), f0=np.concatenate(F0o), f0_t=np.concatenate(F0r), n=np.concatenate(Ne),
                n_t=np.concatenate(Nr), chunks=[], sentences=sents, target=voice_name, donors=[label],
                what=["B3"], modes=dict(f0="-", energy="-", dur="-", dur_scope="-", scope="-", assign="-",
                                        seed=seed, donor_text="-"))
    return dict(block="all", voice=voice_name, ref_voice=None, audio=np.concatenate(audios),
                segments=segments, seg_words=seg_words, seg_altered=seg_altered, pros=pros)


def run(text=TEXT, voice=VOICE, sub_from=SUB_FROM, variants=VARIANTS, prosody_row=PROSODY_ROW,
        geminate=GEMINATE, timing=TIMING, make_reference=MAKE_REFERENCE, normalize=NORMALIZE, speed=SPEED, seed=SEED, out_dir=OUT_DIR):
    from html import escape
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    pack, _ = _voice_pack(voice)
    vname = _name(voice)
    story = load_story(text, name="b3", normalize=normalize, verbose=False)
    ipa = [s for b in story["blocks"] for s in b["ipa"]]
    words = [w for b in story["blocks"] for s in b["spelling"] for w in s]
    jobs = [(vi, v, float(w)) for vi, v in enumerate(variants, start=1) for w in v.get("weights", [0.5])]
    assert len(jobs) <= len(SYMBOLS), f"at most {len(SYMBOLS)} variant × weight combinations per run"
    assert geminate in ("merge", "double", "keep"), geminate
    assert timing in ("original", "model"), timing
    LONG = "ː"

    def sub_map(target):
        """substitution map for SUB_FROM (and its geminate SUB_FROM + ː) -> target symbol"""
        m = {sub_from: target}
        if geminate == "merge":
            m[sub_from + LONG] = target
        elif geminate == "double":
            m[sub_from + LONG] = target + target
        return m

    n_gem = sum(s_.count(sub_from + LONG) for s_ in ipa)
    print(f"{sum(s_.count(sub_from) for s_ in ipa)} × {sub_from}, of which {n_gem} geminate ({sub_from}{LONG}) "
          f"-> geminate mode '{geminate}', timing '{timing}'")
    for v in variants:
        bad = [s for s in v["to"] if s not in model.vocab]
        assert not bad, f"{v.get('label')}: not in Kokoro's vocab: {bad}"

    sections, written = [], []
    # original first (also gives the dashed reference curves)
    refs = []
    for ci, ps in enumerate(ipa):
        if ps.strip():
            _, _, f0, n = render_sentence(ps, pack, speed, seed + ci)
        else:
            f0 = n = np.zeros(2)
        refs.append((f0, n))
    if make_reference:
        res = _render_text(ipa, [None] * len(ipa), refs, pack, vname, "original", speed, seed,
                           orig=ipa, origins=None, timing=timing)
        sections.append(("original", "Original (no changes)", _player_fragment(
            res, words, f"Original · voice {vname}", None, None, "original text")))
        sf.write(out / "b3_original.wav", res["audio"], SR); written.append(out / "b3_original.wav")

    with NewPhonemes() as reg:
        P, A = _tables()
        for k, (vi, v, w) in enumerate(jobs):
            combine = v.get("combine", "interpolate")
            keep_norm = v.get("keep_norm", combine == "interpolate")
            prow = v.get("prosody_row", prosody_row)
            sym = SYMBOLS[k]
            row_ac = make_row(A, sub_from, v["to"], w, combine, keep_norm)
            row_pr = (P[model.vocab[sub_from]].detach().clone() if prow == "from"
                      else make_row(P, sub_from, v["to"], w, combine, keep_norm))
            row_id = reg.add(sym, row_pr, row_ac)
            rule = {"kind": "substitute", "map": sub_map(sym)}
            manip, masks, origins = manipulate_with_quota(ipa, [rule], return_mask=True, return_origin=True,
                                                          verbose=False)
            n_sub = sum(sum(m) for m in masks)
            to_txt = " + ".join(f"{c:g}·{s}" for s, c in v["to"].items())
            how = (f"(1−{w:g})·{sub_from} + {w:g}·({to_txt})" if combine == "interpolate"
                   else f"{sub_from} + {w:g}·({to_txt})")
            info = (f"B3 new phoneme {sym} = {how} · length kept: {keep_norm} · prosody row: {prow} · "
                    f"geminates: {geminate} · timing: {timing} · table row {row_id} · seed {seed} · "
                    f"dashed = original")
            res = _render_text(manip, masks, refs, pack, vname, sym, speed, seed, orig=ipa, origins=origins,
                               timing=timing)
            head = f"V{vi} {v.get('label', '')} · w = {w:g}  [{sym}]"
            stats = block_changes(dict(ipa=ipa, manip=manip), label=f"{sub_from} → {sym} ({how})")
            sections.append((f"v{vi}_w{w:.2f}".replace(".", "_"), head, _player_fragment(
                res, words, f"{head} · voice {vname}", [rule], stats, info)))
            name = f"b3_v{vi}_w{w:.2f}.wav"
            sf.write(out / name, res["audio"], SR); written.append(out / name)
            ac = A[model.vocab[sub_from]].detach()
            print(f"V{vi} w={w:g}: {sym} = {how}, {n_sub} × replaced, cos to {sub_from} "
                  f"{torch.nn.functional.cosine_similarity(row_ac, ac, dim=0).item():.2f}, "
                  f"{len(res['audio']) / SR:.1f} s")
            if combine == "interpolate" and w == 1 and prow == "same" and len(v["to"]) == 1:
                direct = list(v["to"])[0]
                ref_manip, ref_org = manipulate_with_quota(ipa, [{"kind": "substitute", "map": sub_map(direct)}],
                                                           return_origin=True, verbose=False)
                ref_audio = _render_text(ref_manip, [None] * len(ipa), None, pack, vname, direct, speed, seed,
                                         orig=ipa, origins=ref_org, timing=timing)["audio"]
                same = len(ref_audio) == len(res["audio"]) and np.allclose(ref_audio, res["audio"], atol=1e-3)
                print(f"   self-check: identical to writing '{direct}' in the text directly: {same}"
                      + ("" if same else "   <-- the new symbol is NOT read as intended (check _encode's symbol map)"))

    page = out / f"b3_{sub_from}_{TO}_variants.html"
    title = f"B3 new phonemes for {sub_from} · voice {vname}"
    intro = escape(f"Every {sub_from} in the text replaced by a new symbol whose table rows are built from "
                   f"existing ones (see each player's description line). No extra tokens: timing comes from the "
                   f"model. Pitch panel: solid = this version, dashed = original.")
    page.write_text(_page(title, intro, sections), encoding="utf-8")
    written.insert(0, page)
    print(f"written to {out}/ :")
    for p in written:
        print("   ", Path(p).name)
    return written


if __name__ == "__main__":
    run()

56 × t, of which 15 geminate (tː) -> geminate mode 'merge', timing 'original'
V1 w=0.6: Ⓐ = (1−0.6)·t + 0.6·(1·ɖ), 56 × replaced, cos to t 0.52, 26.6 s
V1 w=0.7: Ⓑ = (1−0.7)·t + 0.7·(1·ɖ), 56 × replaced, cos to t 0.38, 26.6 s
V1 w=0.8: Ⓒ = (1−0.8)·t + 0.8·(1·ɖ), 56 × replaced, cos to t 0.25, 26.6 s
V1 w=0.9: Ⓓ = (1−0.9)·t + 0.9·(1·ɖ), 56 × replaced, cos to t 0.14, 26.6 s
V1 w=1: Ⓔ = (1−1)·t + 1·(1·ɖ), 56 × replaced, cos to t 0.04, 26.6 s
   self-check: identical to writing 'ɖ' in the text directly: True
written to b3_output/ :
    b3_t_ɖ_variants.html
    b3_original.wav
    b3_v1_w0.60.wav
    b3_v1_w0.70.wav
    b3_v1_w0.80.wav
    b3_v1_w0.90.wav
    b3_v1_w1.00.wav
